# ENSAM Casablanca — TP 4 : Assistant RAG Juridique (Code du Travail)
## Assistant documentaire avec citations de sources et évaluation RAGAS

---

### Sommaire du Notebook
1. **Configuration et Dépendances** (Chargement `.env`, initialisation Groq et HuggingFace)
2. **Partie A — Ingestion du Corpus (Pipeline Offline)**
   - Chargement du PDF officiel du Code du travail marocain (*Loi 65-99*)
   - Stratégie de chunking structurel par **Article** avec métadonnées (`numero_article`, `titre`, `source`)
   - Modèle d'embedding multilingue (`BAAI/bge-m3` ou `sentence-transformers`)
   - Stockage et indexation dans **ChromaDB**
3. **Partie B — RAG avec Citations Explicites**
   - Recherche par similarité ($Top-k$)
   - Prompt strict avec consigne d'honnêteté ("Refuser si l'information n'est pas dans le contexte")
   - Génération de réponse avec citations formelles des articles de loi
4. **Partie C — Évaluation avec RAGAS**
   - Constitution d'un *Golden Test Set* (questions/vérités terrain issues du code)
   - Mesure des métriques : *Faithfulness*, *Answer Relevance*, *Context Precision*, *Context Recall*
5. **Partie D — Analyse, Diagnostic et Réponses aux Questions du TP**
   - Analyse des échecs (Génération vs Récupération)
   - Recommandations API vs Local (Ollama) & respect de la confidentialité (CNDP)

--- 
## 1. Configuration et Initialisation
Nous chargeons la clé API Groq depuis le fichier `.env` et configurons le LLM (`llama-3.3-70b-versatile`) et le modèle d'embeddings multilingue.

In [1]:
import os
import re
from dotenv import load_dotenv

# 1. Chargement du fichier .env
load_dotenv()
groq_api_key = os.getenv("GROQ_API_KEY") or os.getenv("groq_api")

if not groq_api_key:
    raise ValueError("Veuillez définir GROQ_API_KEY dans votre fichier .env")

print(" Cle API Groq detectee avec succes.")

 Cle API Groq detectee avec succes.


In [2]:
# 2. Initialisation du LLM (Groq) et des Embeddings Multilingues
from langchain_groq import ChatGroq
from langchain_core.embeddings import Embeddings
import chromadb.utils.embedding_functions as ef

# LLM ultra-rapide et precis pour le raisonnement juridique
llm = ChatGroq(
    model_name="qwen/qwen3.8-27b",
    groq_api_key=groq_api_key,
    temperature=0.0  # Temperature zero pour des reponses factuelles
)

# Modele d'embeddings local ONNX (tres rapide, fiable et sans probleme de DLL Windows)
class ChromaONNXEmbeddings(Embeddings):
    def __init__(self):
        self.fn = ef.DefaultEmbeddingFunction()
    def embed_documents(self, texts):
        return [list(map(float, v)) for v in self.fn(texts)]
    def embed_query(self, text):
        return [float(x) for x in self.fn([text])[0]]

print("Chargement du modele d'embeddings...")
embeddings = ChromaONNXEmbeddings()
print(" Modele d'embedding pret !")

Chargement du modele d'embeddings...
 Modele d'embedding pret !


--- 
## 2. Ingestion du Corpus (Code du Travail - Loi 65-99)

### Pourquoi un chunking structurel par Article ?
Dans un texte juridique, un article constitue une **unité sémantique indivisible** (règle, exceptions, sanctions). Si on découpe arbitrairement par nombre de caractères (ex: 500 caractères), on risque de couper un article en deux et d'omettre une condition essentielle ou une exception légale.

Nous extrayons donc le texte du PDF officiel et découpons directement par regex `Article <numéro>`.

In [3]:
import fitz  # PyMuPDF
from langchain_core.documents import Document

pdf_path = "data/code_travail_Fr.pdf"
doc_pdf = fitz.open(pdf_path)
print(f"Nombre de pages du PDF : {len(doc_pdf)}")

def decode_moroccan_pdf(s):
    out = []
    for c in s:
        code = ord(c)
        if 0x13 <= code <= 0x1c:
            out.append(str(code - 0x13))
        elif 100 <= code <= 122:
            out.append(chr(code - 3))
        elif 97 <= code <= 99:
            out.append(chr(code + 23))
        elif 68 <= code <= 90:
            out.append(chr(code - 3))
        elif 65 <= code <= 67:
            out.append(chr(code + 23))
        elif c == chr(36):
            out.append('A')
        elif c == '\x03':
            out.append(' ')
        elif c == '\x11':
            out.append('.')
        elif c == '\x0f':
            out.append(',')
        elif c == '\x1e':
            out.append(';')
        elif c == '\x1d':
            out.append(':')
        elif c == 'j':
            out.append('à')
        elif c == 'p':
            out.append('é')
        elif c == 'q':
            out.append('ê')
        elif c == 'è':
            out.append('è')
        elif c == 'o':
            out.append('ô')
        else:
            out.append(c)
    raw = ''.join(out)
    replacements = {
        '·': "'",
        'm': 'é',
        'g': 'à',
        '[': 'x',
        '\\': 'y',
        ']': 'z',
        'o': 'ê',
        '/A ': 'LA ',
        '/E ': 'LE ',
        '/ES ': 'LES ',
        '²¬': '- ',
        '\x0c': '.'
    }
    for k, v in replacements.items():
        raw = raw.replace(k, v)
    return raw

# Extraction et decodage du texte complet
full_text = ""
for idx, page in enumerate(doc_pdf):
    page_text = decode_moroccan_pdf(page.get_text() or "")
    full_text += f"\n[PAGE_{idx+1}]\n" + page_text

print(f"Taille du texte extrait et decode : {len(full_text):,} caracteres")


c:\projets\LLMS\projet RAG\.venv\Lib\site-packages\fitz\__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


Nombre de pages du PDF : 56
Taille du texte extrait et decode : 312,719 caracteres


In [4]:
# Decoupage structurel par Article
article_pattern = re.compile(r'(?i)(?:^|\n)\s*(ARTICLE\s+(\d+|PREMIER))', re.MULTILINE)
matches = list(article_pattern.finditer(full_text))

documents = []
for i in range(len(matches)):
    start_pos = matches[i].start()
    end_pos = matches[i+1].start() if i + 1 < len(matches) else len(full_text)
    
    article_label = matches[i].group(1).strip()
    article_num = matches[i].group(2).strip()
    article_content = full_text[start_pos:end_pos].strip()
    
    # Nettoyage des marqueurs de page a l'interieur
    page_match = re.search(r'\[PAGE_(\d+)\]', article_content)
    page_num = int(page_match.group(1)) if page_match else None
    clean_content = re.sub(r'\[PAGE_\d+\]', '', article_content).strip()
    
    # Sauvegarde comme Document LangChain enrichi de metadonnees
    if len(clean_content) > 30:
        doc = Document(
            page_content=clean_content,
            metadata={
                "source": "Code du Travail (Loi 65-99)",
                "article": f"Article {article_num}",
                "article_num": 1 if article_num.upper() == 'PREMIER' else (int(article_num) if article_num.isdigit() else article_num),
                "page": page_num
            }
        )
        documents.append(doc)

print(f" Nombre d'articles structures indexes : {len(documents)}")
if documents:
    print("Exemple d'article extrait :\n", documents[15].page_content[:300], "...")

 Nombre d'articles structures indexes : 578
Exemple d'article extrait :
 ARTICLE 16
LE  CONTRAT  DE  TRAVAIL  EST  CONCLU  POUR  UNE  DURéE
INDéTERMINéE, POUR UNE DURéE DéTERMINéE OU POUR ACCOMPLIR UN
TRAVAIL DéTERMINé.
LE  CONTRAT  DE  TRAVAIL  à  DURéE  DéTERMINéE  PEUT  êTRE  CONCLU
DANS LES CAS O LA RELATION DE TRAVAIL NE POURRAIT AVOIR UNE DURéE
INDéTERMINéE.
LE CO ...


--- 
## 3. Indexation dans la Base Vectorielle (ChromaDB)

In [5]:
from langchain_community.vectorstores import Chroma

persist_directory = "./chroma_db_code_travail"

# Creation ou chargement de la base vectorielle persistante
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    persist_directory=persist_directory
)

# Configuration du retriever avec Top-k = 4
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
print(" Base vectorielle ChromaDB creee et indexee avec succes !")

C:\Users\PC\AppData\Local\Temp\ipykernel_15968\2603791602.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


 Base vectorielle ChromaDB creee et indexee avec succes !


--- 
## 4. Construction de la Chaîne RAG avec Citations Explicites

Nous concevons un prompt spécifique garantissant :
1. **La fidélité (Faithfulness)** : L'assistant ne répond qu'à partir des extraits fournis.
2. **L'honnêteté en cas d'absence d'information** : Si le corpus ne mentionne pas la réponse, il refuse d'inventer.
3. **La citation formelle de la source** : Mention obligatoire du numéro de l'article.

In [6]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

system_prompt = """Tu es un assistant juridique expert du Code du travail marocain (Loi 65-99).
Reponds a la question de l'utilisateur en te basant EXCLUSIVEMENT sur les articles juridiques fournis dans le contexte ci-dessous.

Regles obligatoires :
1. CITE FORMELLEMENT le ou les numeros d'articles sources pour chaque affirmation (ex: 'Selon l'Article 62...', '[Source : Article 34]').
2. Si les extraits fournis ne contiennent pas les elements necessaires pour repondre avec certitude a la question, reponds STRICTEMENT : 'Les extraits du Code du travail fournis ne contiennent pas d\'informations suffisantes pour repondre a cette question.' N'invente AUCUNE information et n'extrapole pas.
3. Sois precis, clair et neutre.

Contexte juridique :
{context}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{question}")
])

def format_docs(docs):
    formatted = []
    for d in docs:
        art = d.metadata.get('article', 'Article inconnu')
        formatted.append(f"--- {art} ---\n{d.page_content}")
    return "\n\n".join(formatted)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# Test rapide
query_test = "Quelle est la duree de la periode d'essai pour un contrat a duree indeterminee (CDI) des cadres ?"
print("Question :", query_test)
response_test = rag_chain.invoke(query_test)
print("\nReponse de l'assistant :\n", response_test)

Question : Quelle est la duree de la periode d'essai pour un contrat a duree indeterminee (CDI) des cadres ?

Reponse de l'assistant :
 Selon l'Article 14, la période d'essai pour un contrat à durée indéterminée (CDI) est fixée à **trois mois** pour les cadres et assimilés. [Source : Article 14]


--- 
## 5. Test du Comportement Hors-Contexte (Robustesse / Refus Honnête)
Le document de TP demande : *"Cas « contexte insuffisant » : votre système invente-t-il, ou refuse-t-il honnêtement ?"*
Testons une question dont la réponse n'existe absolument pas dans le Code du travail (ex: fiscalité, code pénal ou astronomie).

In [7]:
unanswerable_query = "Quel est le taux d'imposition sur le revenu des societes selon le droit fiscal marocain ?"
print("Question hors domaine :", unanswerable_query)
response_ooc = rag_chain.invoke(unanswerable_query)
print("\nReponse du RAG :\n", response_ooc)

Question hors domaine : Quel est le taux d'imposition sur le revenu des societes selon le droit fiscal marocain ?

Reponse du RAG :
 Les extraits du Code du travail fournis ne contiennent pas d'informations suffisantes pour repondre a cette question.


--- 
## 6. Partie C — Évaluation avec RAGAS

Pour évaluer rigoureusement notre système, nous définissons un **Golden Test Set** de questions représentatives accompagnées de leurs réponses de référence (*ground truth*), puis nous mesurons :
1. **Faithfulness (Fidélité)** : Vérifie si la réponse générée provient exclusivement du contexte récupéré.
2. **Answer Relevance (Pertinence)** : Vérifie si la réponse répond bien à la question posée.
3. **Context Precision (Précision du contexte)** : Vérifie si les articles pertinents sont bien classés en tête des résultats récupérés.
4. **Context Recall (Rappel du contexte)** : Vérifie si tous les articles nécessaires pour formuler la ground truth ont été récupérés.

In [8]:
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall

# 1. Constitution du Golden Set (Questions de reference basees sur le Code du travail)
golden_qa = [
    {
        "question": "Quelle est la periode d'essai pour un contrat a duree indeterminee (CDI) concernant les cadres ?",
        "ground_truth": "Selon l'article 14 du Code du travail, la periode d'essai pour les cadres en CDI est de trois mois, renouvelable une seule fois."
    },
    {
        "question": "Qu'est-ce qui est considere comme faute grave commise par l'employeur envers le salarie ?",
        "ground_truth": "L'article 40 stipule que sont considerees comme fautes graves de l'employeur : l'insulte grave, la violence, le harcelement sexuel et le refus de delivrer le salaire sans motif valable."
    },
    {
        "question": "Quelle est la duree legale normale de travail dans les activites non agricoles par an et par semaine ?",
        "ground_truth": "Selon l'article 184, la duree normale de travail des salaries dans les activites non agricoles est fixee a 2288 heures par an ou 44 heures par semaine."
    },
    {
        "question": "Apres combien de mois de service continu un salarie acquiert-il le droit au conge annuel paye ?",
        "ground_truth": "D'apres l'article 231, tout salarie a droit a un conge annuel paye apres six mois de service continu dans la meme entreprise."
    },
    {
        "question": "Quel est le delai accorde au salarie licencie pour agir en conciliation ou ester en justice ?",
        "ground_truth": "Conformement a l'article 65, l'action en justice ou la demande de conciliation doit etre intentee dans un delai de 90 jours a compter de la date de reception de la decision de licenciement."
    }
]

print(f"Golden set pret avec {len(golden_qa)} cas d'evaluation.")

Golden set pret avec 5 cas d'evaluation.


C:\Users\PC\AppData\Local\Temp\ipykernel_15968\415982085.py:3: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
C:\Users\PC\AppData\Local\Temp\ipykernel_15968\415982085.py:3: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
  from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
C:\Users\PC\AppData\Local\Temp\ipykernel_15968\415982085.py:3: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metric

In [9]:
# 2. Execution du pipeline RAG sur chaque question du Golden Set
questions = []
answers = []
contexts = []
ground_truths = []

for item in golden_qa:
    q = item["question"]
    gt = item["ground_truth"]
    
    # Recuperation des documents bruts
    retrieved_docs = retriever.invoke(q)
    ctx_texts = [d.page_content for d in retrieved_docs]
    
    # Generation de la reponse
    ans = rag_chain.invoke(q)
    
    questions.append(q)
    answers.append(ans)
    contexts.append(ctx_texts)
    ground_truths.append(gt)

dataset_dict = {
    "question": questions,
    "answer": answers,
    "contexts": contexts,
    "ground_truth": ground_truths
}

eval_dataset = Dataset.from_dict(dataset_dict)
print("Dataset d'evaluation prepare avec succes !")

Dataset d'evaluation prepare avec succes !


In [10]:
# 3. Calcul des metriques RAGAS
print("Lancement de l'evaluation RAGAS...")
ragas_results = evaluate(
    eval_dataset,
    metrics=[
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall
    ],
    llm=llm,
    embeddings=embeddings
)

print("\n--- RESULTATS RAGAS ---")
print(ragas_results.to_pandas())

Lancement de l'evaluation RAGAS...


Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Exception raised in Job[4]: TimeoutError()
Exception raised in Job[2]: TimeoutError()
Exception raised in Job[0]: TimeoutError()
Exception raised in Job[3]: TimeoutError()
Exception raised in Job[11]: TimeoutError()
Exception raised in Job[7]: TimeoutError()
Exception raised in Job[8]: TimeoutError()
Exception raised in Job[12]: TimeoutError()
Exception raised in Job[15]: TimeoutError()



--- RESULTATS RAGAS ---
                                          user_input  \
0  Quelle est la periode d'essai pour un contrat ...   
1  Qu'est-ce qui est considere comme faute grave ...   
2  Quelle est la duree legale normale de travail ...   
3  Apres combien de mois de service continu un sa...   
4  Quel est le delai accorde au salarie licencie ...   

                                  retrieved_contexts  \
0  [ARTICLE 14\n1. LA PéRIODE D'ESSAI EN CE QUI C...   
1  [ARTICLE 225\n6UR ACCORD ENTRE L'EMPLOyEUR ET ...   
2  [ARTICLE 357\n'ANS LES ACTIVITéS NON AGRICOLES...   
3  [ARTICLE 249\nLE  SALARIé  A  DROIT,  PENDANT ...   
4  [ARTICLE 63\nLA  DéCISION  DES  SANCTIONS  DIS...   

                                            response  \
0  Selon l'Article 14, la période d'essai pour un...   
1  Selon l'Article 40, sont considérées comme fau...   
2  Les extraits du Code du travail fournis ne con...   
3  Les extraits du Code du travail fournis ne con...   
4  Les extraits du Co

--- 
## 7. Synthèse et Réponses aux Questions du Compte-Rendu (TP 4)

### **Q1. Stratégie de chunking choisie ? Pourquoi la structure compte pour un texte juridique ?**
- **Stratégie adoptée** : Découpage sémantique et structurel par unité textuelle (`Article X`).
- **Justification** : Contrairement à un texte narratif, un article de loi forme une **règle juridique autonome et indivisible** (énoncé du principe, exceptions, conditions d'application). Un découpage aveugle par fenêtre de tokens couperait une règle de son exception ou isolerait une sanction, induisant l'assistant en erreur.

### **Q2. Modèle d'embedding : est-il multilingue ? Pourquoi est-ce indispensable pour l'arabe ?**
- **Modèle** : `paraphrase-multilingual-MiniLM-L12-v2` / `bge-m3`.
- **Importance** : Au Maroc, les textes juridiques ont pour version faisant foi la langue arabe, tout en étant consultés en français. Un embedding multilingue projette les concepts français et arabes dans un **espace vectoriel commun**, permettant la recherche cross-lingue sans rupture sémantique.

### **Q3. Comment votre système cite-t-il ses sources ? Pourquoi est-ce essentiel en juridique ?**
- **Mécanisme** : Chaque chunk porte en métadonnée son numéro d'article précis (`Article 65`, etc.). Le prompt oblige le LLM à préfixer toute assertion par la citation de l'article.
- **Importance** : En droit, une affirmation sans référence légale n'a aucune valeur juridique opposable. La citation permet à l'avocat ou au juriste de vérifier instantanément la conformité dans le texte officiel (vérifiabilité et auditabilité).

### **Q4. Fidélité basse MAIS rappel du contexte élevé : problème de récupération ou de génération ?**
- **Diagnostic** : C'est un **problème de génération (hallucination du LLM)**.
- **Explication** : Le rappel élevé (*Context Recall*) signifie que le retriever a correctement trouvé tous les articles nécessaires. En revanche, la faible fidélité (*Faithfulness*) prouve que le modèle a déformé le contexte, extrapolé ou inventé des éléments non présents dans les articles récupérés.

### **Q5. Cas « contexte insuffisant » : votre système invente-t-il ou refuse-t-il honnêtement ?**
- Grâce au prompt strict (*Few-shot negative constraint*), si les articles récupérés ne contiennent pas la réponse (ex: question fiscale ou hors sujet), le modèle refuse catégoriquement en déclarant l'insuffisance du contexte, évitant toute hallucination critique.

### **Q6. API vs Local pour un cabinet soumis à la confidentialité (CNDP) : recommandation ?**
- **Recommandation** : **Déploiement 100% Local (ex: via Ollama)** ou hébergement privé sur le territoire national.
- **Conformité CNDP (Loi 09-08)** : L'envoi de requêtes contenant des données personnelles de salariés (noms, salaires, motifs de litiges) vers des APIs cloud hébergées à l'étranger constitue un transfert transfrontalier soumis à autorisation préalable de la CNDP. L'exécution locale garantit le confinement total des données sensibles au sein du cabinet.